# Phase 2 training on Colab
Trains the normal-only autoencoders (dense baseline and convolutional) for all 8 machine IDs.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Upload `phase2_bundle.zip` to Google Drive, folder `My Drive/smart_listener/`.

Run the cells top to bottom. If Colab disconnects, run again from the top: finished machines are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile, shutil, torch
DRIVE = '/content/drive/MyDrive/smart_listener'
BUNDLE = f'{DRIVE}/phase2_bundle.zip'
RESULTS = f'{DRIVE}/phase2_results.zip'
assert os.path.exists(BUNDLE), 'phase2_bundle.zip not found in My Drive/smart_listener/'
shutil.rmtree('/content/proj', ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall('/content/proj')
# resume: bring back results from an earlier session, if any
if os.path.exists(RESULTS):
    zipfile.ZipFile(RESULTS).extractall('/content/proj')
    print('restored earlier results')
print('GPU available:', torch.cuda.is_available())
print(os.listdir('/content/proj'))

In [ ]:
!pip -q install soundfile librosa pyyaml scikit-learn pandas matplotlib

In [ ]:
%cd /content/proj
%env PYTHONPATH=/content/proj/src

In [ ]:
def save_results():
    import zipfile, glob
    with zipfile.ZipFile(RESULTS, 'w', zipfile.ZIP_DEFLATED) as z:
        for pat in ['models/ad/**/*', 'reports/phase2/*', 'reports/figures/phase2_*.png']:
            for p in glob.glob(pat, recursive=True):
                if os.path.isfile(p):
                    z.write(p)
    print('saved to', RESULTS)

## 1. Quick smoke test (about 1 minute)

In [ ]:
!python scripts/train_ad.py --smoke
!python scripts/evaluate_ad.py --smoke

## 2. Full training, fans first, then pumps
Each cell saves its results to Drive when it finishes.

In [ ]:
!python scripts/train_ad.py --groups fan_id_00 fan_id_02 fan_id_04 fan_id_06
save_results()

In [ ]:
!python scripts/train_ad.py --groups pump_id_00 pump_id_02 pump_id_04 pump_id_06
save_results()

## 3. Evaluate

In [ ]:
!python scripts/evaluate_ad.py
save_results()

Finished. Download `phase2_results.zip` from Drive and extract it into your project folder on the laptop (it contains `models/ad`, `reports/phase2` and the figures).